In [0]:
raw_path = "/Volumes/f1_catalog/bronze/raw/drivers.csv"

drivers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_path)
)

# display(drivers_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

watermark_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("processed_timestamp", TimestampType(), False)
])

empty_df = spark.createDataFrame([], watermark_schema)

empty_df.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("f1_catalog.gold.pipeline_watermark")

In [0]:
# spark.sql("SELECT * FROM f1_catalog.gold.pipeline_watermark").show()

In [0]:
from pyspark.sql import functions as F

def ingest_csv(file_name, table_name):

    source_path = f"/Volumes/f1_catalog/bronze/raw/{file_name}"
    watermark_table = "f1_catalog.gold.pipeline_watermark"
    bronze_table = f"f1_catalog.bronze.{table_name}"

    processed = (
        spark.table(watermark_table)
        .filter(
            (F.col("table_name") == table_name) &
            (F.col("file_name") == file_name)
        )
        .limit(1)
        .count()
    )

    if processed > 0:
        print(f"Skipping {file_name} - already processed")
        return

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
    )

    df = (
        df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_name))
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    watermark_df = spark.createDataFrame(
        [(table_name, file_name)],
        ["table_name", "file_name"]
    ).withColumn(
        "processed_timestamp",
        F.current_timestamp()
    )

    (
        watermark_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(watermark_table)
    )

    print(f"Loaded {file_name} → {bronze_table}")